In [ ]:
# 사전 정의된 헬퍼 (my_fsr_cmd, 그래프 유틸 등)를 로드
%run My_script.ipynb

import chipwhisperer as cw

# ─────────────────────────────────────────
# 타겟 / 펌웨어 관련 상수
# ─────────────────────────────────────────
PLATFORM      = 'CW308_STM32F3'   # 타겟 보드 종류
SCOPETYPE     = 'OPENADC'         # 캡처 장치 (Lite/Husky 공통)
CRYPTO_TARGET = 'NONE'            # 사용 암호 라이브러리 (없음 -> 자체 펌웨어)
SS_VER        = 'SS_VER_2_1'      # SimpleSerial 프로토콜 버전

In [ ]:
def connect_all_devices() -> dict:
    """연결된 모든 ChipWhisperer 장치에 접속하여 딕셔너리로 반환"""
    device_list = cw.list_devices()

    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다.")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}

    for device in device_list:
        # 'ChipWhisperer-Lite' → 'ChipWhisperer_Lite' 처럼 dict 키로 쓰기 쉽게 변환
        name = device['name'].replace("-", "_")
        sn   = device['sn']

        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료  (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패  (SN: {sn})\n      └─ {e}")

    return scopes

# PC에 연결된 모든 장치 인식 및 할당
scopes = connect_all_devices()
lite_scope = scopes["ChipWhisperer_Lite"]
husky_scope = scopes["ChipWhisperer_Husky"]

In [ ]:
# Lite를 통한 타겟 보드 연결 설정
if SS_VER == "SS_VER_2_1":
    target_type = cw.targets.SimpleSerial2
else:
    raise OSError("지원되지 않는 SimpleSerial 버전입니다.")

try:
    target = cw.target(lite_scope, target_type)
    print("\n[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공")
except Exception as e:
    print(f"\n[✗] ChipWhisperer_Lite에 타겟 보드 연결 실패: {e}")

In [ ]:
# 1. 펌웨어 컴파일 (서브프로세스를 통한 bash 명령 대체)
print("펌웨어 컴파일 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 완료")

# 2. 프로그래머 설정
if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    raise OSError("프로그래머가 지원되지 않는 플랫폼입니다.")

# 3. 펌웨어 플래싱 (Lite가 프로그래머 역할을 수행)
lite_scope.default_setup()
try:
    hex_path = f"simpleserial_main/simpleserial-base-{PLATFORM}.hex"
    cw.program_target(lite_scope, prog, hex_path)
    print(f"[✓] {PLATFORM} 타겟 보드에 프로그램 업로드 완료")
except Exception as e:
    print(f"[✗] 펌웨어 프로그램 실패: {e}")

# 4. 펌웨어 컴파일 클린 (서브프로세스를 통한 bash 명령 대체)
print("펌웨어 컴파일 클린 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}", "clean"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 클린 완료")

In [ ]:
MAX_DATA_LEN = 100

# 재현성을 위해 시드 고정
random.seed(1)

# 무작위 키(k), 평문(p) 생성 후 호스트에서 사전 계산한 골든 결과(k XOR p)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

# 0x81 = 데이터 전송 명령 ('k'=key, 'p'=plaintext, 'l'=length)
# 0x82 = 연산 트리거 명령
# 0x83 = 결과 회수 명령
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
print(f'타겟 결과 : {Return_k_XOR_p.hex(" ")}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print()
if Golden_k_XOR_p == Return_k_XOR_p:
    print('[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
else:
    print('[✗] 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')

In [ ]:
# 이전 설정의 간섭을 방지하기 위해 Husky 공장 초기화
husky_scope.default_setup()
time.sleep(0.5)

## 전압 글리치 모드 활성 코드
husky_scope.vglitch_setup('both')   
if husky_scope._is_husky:
    husky_scope.vglitch_setup('hp', default_setup=False) # HP alone works best for Husky
else:
    husky_scope.vglitch_setup('both', default_setup=False) # use both transistors
time.sleep(0.5)

print("Husky 스코프 하드웨어 동기화 및 튜닝 중...")

# ---------------------------------------------------------
# [1] 클럭 신호 탐색 및 동기화 (Aux in/out)
# ---------------------------------------------------------
husky_scope.clock.clkgen_freq = 0
husky_scope.clock.reset_adc()
# AUX MCX 를 입력(high-Z)으로 설정 → Husky 가 클럭을 driving 하지 않고 수신만 함
husky_scope.io.aux_io_mcx = 'high_z'
# PLL 입력 소스를 외부 클럭(extclk)으로 지정
husky_scope.clock.clkgen_src = 'extclk_aux_io'
husky_scope.clock.reset_adc()

if (husky_scope.io.aux_io_mcx == 'high_z') and (husky_scope.clock.clkgen_src == 'extclk_aux_io'):
    print(f"[✓] io.aux_io_mcx     = {husky_scope.io.aux_io_mcx}")
    print(f"[✓] clock.clkgen_src  = {husky_scope.clock.clkgen_src}")
else:
    print(f"[✗] 외부 클럭 설정 실패")

In [ ]:
# ---------------------------------------------------------
# [2] 외부 클럭 주파수 탐색 및 동기화
# ---------------------------------------------------------
# 더 정밀한 클럭 주파수 설정 가능 (동기화 및 PLL 잠금이 실패할 경우에 주석처리하면 동작할 수 있습니다)
husky_scope.clock.pll._allow_rdiv = True
# 주파수 카운터의 측정 대상을 외부 클럭으로 지정
husky_scope.clock.freq_ctr_src = 'extclk'
# 카운터 초기 안정화를 위한 짧은 대기
time.sleep(0.5)

# 주파수 데이터 수집 (0.2초 간격, 20회)
freqs = []
for _ in range(20):
    freqs.append(husky_scope.clock.freq_ctr)
    time.sleep(0.2)
data = pd.Series(freqs)
print(f"최빈값: {data.mode().iloc[0]} (등장 {(data == data.mode().iloc[0]).sum()}/{len(data)}회)")
print(f"범위: {data.min()} ~ {data.max()} (Δ={data.max()-data.min()})")
print("\n[전체 통계 요약]")
print(data.describe()) # 개수, 평균, 표준편차, 최소, 최대, 사분위수 출력

# 내부/외부 클럭 주파수 동기화
husky_scope.clock.clkgen_freq = data.mode().iloc[0]
# ADC 샘플레이트 클럭 동기화: 1 샘플 = 1 클럭
husky_scope.clock.adc_mul    = 1            # 오버샘플링 없음
husky_scope.adc.decimate     = 1            # 다운샘플링 없음
# ADC 리셋
husky_scope.clock.reset_adc()

if husky_scope.clock.adc_locked:
    print("[✓] ADC 클럭 동기화 완료")
    print(f"   - ADC 샘플레이트 (adc_freq): {husky_scope.clock.adc_freq:,.0f} Hz")
else:
    print("[✗] ADC 클럭 동기화 실패 (Lock Error)")

if husky_scope.clock.clkgen_locked:
    print("[✓] Husky PLL 잠금 성공")
    print(f"   - 타겟 클럭 (clkgen_freq) : {husky_scope.clock.clkgen_freq:,.0f} Hz")
else:
    print("[✗] Husky PLL 잠금 실패! 외부 클럭의 진폭/듀티/안정성을 확인하세요.")

In [ ]:
# ---------------------------------------------------------
# [3] 트리거 핀 설정 (전면 USERIO - D0 핀)
# ---------------------------------------------------------
# 트리거 입력 소스 = 전면 USERIO D0
husky_scope.trigger.triggers = 'userio_d0'
# 단순 엣지/레벨 검출용 'basic' 트리거 모듈 사용
husky_scope.trigger.module = 'basic'
# 캡처 시작 조건 = 상승 엣지 (타겟이 트리거를 LOW → HIGH 로 토글)
husky_scope.adc.basic_mode = 'rising_edge'

print("[✓] Husky 스코프 파라미터 설정 완료")
print(f"trigger.triggers = {husky_scope.trigger.triggers}")
print(f"trigger.module   = {husky_scope.trigger.module}")
print(f"adc.basic_mode   = {husky_scope.adc.basic_mode}")

In [ ]:
# LNA 게인 (dB). 보통 20~30 dB 사이. 너무 높으면 클리핑, 너무 낮으면 SNR 저하.
husky_scope.gain.db = 25

# 한 번의 캡처에서 수집할 샘플 개수
husky_scope.adc.samples = 2000

# 트리거 이후 캡처 시작점 (0 = 트리거 즉시 캡처 시작)
husky_scope.adc.offset = 0

# 트리거 이전 샘플 (사전 캡처). 필요 시 양수로 설정 가능.
husky_scope.adc.presamples = 0

print(f"gain.db          = {husky_scope.gain.db}")
print(f"adc.samples      = {husky_scope.adc.samples}")
print(f"adc.offset       = {husky_scope.adc.offset}")
print(f"adc.presamples   = {husky_scope.adc.presamples}")

In [ ]:
# ── 글리치 발생 시점 + 출력 모드 ────────────────────────
husky_scope.glitch.arm_timing = 'after_scope'   # scope arm 후 글리치 활성
husky_scope.glitch.output     = 'glitch_only'   # · glitch_only : 전압 글리치용 

# ── 경고 로그 무시 (탐색 중 빈번하게 발생하는 정상 경고) ──
# (ChipWhisperer Target  WARNING) Read timed out
# (ChipWhisperer Glitch  WARNING) Partial reconfiguration for width = 0 may not work
logging.getLogger('ChipWhisperer Target').setLevel(logging.ERROR)
logging.getLogger('ChipWhisperer Glitch').setLevel(logging.ERROR)

print('[✓] 글리치 출력 모드 설정 완료')
print(f'  arm_timing : {husky_scope.glitch.arm_timing}')
print(f'  output     : {husky_scope.glitch.output}')

In [ ]:
def Encrypt(data_k, data_p):    
    my_fsr_cmd(target, 0x81, 'k', data_k)
    my_fsr_cmd(target, 0x81, 'p', data_p)
    my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
    my_fsr_cmd(target, 0x82, 'c', [])  
    return my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

# 수집 데이터 초기화
t_husky = []
i_k = []
i_p = []
o_c = []

# 고정 시드
random.seed(1)
MAX_DATA_LEN = 100  # 한 번에 전송 가능한 최대 데이터 크기 (바이트)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))


husky_scope.glitch.repeat = 5                   # 하나의 오류주입에 버스트 횟수
husky_scope.glitch.num_glitches = 1             # 하나의 암호화 시행에 오류가 주입되는 클럭 개수

for i_ext_offset in trange(170, 172, desc='i_ext_offset', leave=False):

    #for i_offset in range(0, husky_scope.glitch.phase_shift_steps, 500):
    for i_offset in range(0, 500, 50):
        
        #for i_width in range(0, husky_scope.glitch.phase_shift_steps // 2, 500):
        for i_width in range(1900, 2200, 50):
            # 유효성 검사 — 의미 없는 파라미터 건너뛰기
            if i_width == 0:
                continue
            if (i_offset + i_width) > husky_scope.glitch.phase_shift_steps:
                continue

            # 글리치 파라미터 설정
            husky_scope.glitch.ext_offset = i_ext_offset    # 트리거 후 N 클럭
            husky_scope.glitch.offset     = i_offset        # 1 클럭 내 시작 위상
            husky_scope.glitch.width      = i_width         # 글리치 펄스 폭
            for _ in range(10):
                try:
                    reset_target(lite_scope)
                    husky_scope.arm()
                    ct = Encrypt(data_k, data_p)
                    ret_husky = husky_scope.capture()
                    if ret_husky:
                        print(".")
                        continue
                except:
                    print("..")
                    continue  

                if Golden_k_XOR_p == ct:
                    continue    
                    
                wave_husky = husky_scope.get_last_trace()

                t_husky.append(wave_husky)
                i_k.append(data_k)
                i_p.append(data_p)
                o_c.append(ct)  

In [ ]:
print(f'타겟 결과 : {o_c[0].hex(" ")}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print(o_c[2] != Golden_k_XOR_p)

In [ ]:
plot_t(t_husky, num_plot=len(t_husky))
print('\n[✓] 시각화 완료!')

In [ ]:
def disconnect_all_devices(scopes: dict) -> None:
    # 타겟 객체 먼저 닫기 (Lite 의 UART 점유 해제)
    try:
        target.dis()
        print("  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료")
    except Exception as e:
        print(f"  [✗] 타겟 보드 연결 해제 실패  └─ {e}")

    """딕셔너리 내 모든 장치 연결 해제"""
    print("\n장치 연결 해제 중...")
    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")
    scopes.clear()

# 파형 수집 및 데이터 저장이 끝난 후 반드시 포트 및 메모리 자원 반환
disconnect_all_devices(scopes)